# QLoRA Qwen2.5-7B trên GPU 12GB, so với model nhỏ và với prompt

Bài Intermediate 03 fine-tune model 0.5B cho bài trích xuất phản hồi UIT-VSFC. Câu hỏi của bài này: model **7B** fine-tune
bằng QLoRA (trọng số NF4 + double quantization của module 10, LoRA bf16, optimizer phân trang) có đáng gấp nhiều lần
VRAM và thời gian không, so với 7B chỉ dùng prompt và với 0.5B fine-tune trên **cùng** dữ liệu? Bạn ước lượng bộ nhớ
trước khi chạy, đo VRAM đỉnh và thời gian thật, rồi viết `REPORT.md`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; dán `to_chat`, `parse_output`, `macro_f1`,
  `generate_batch` của bạn từ bài Intermediate 03.
- Cell `explore` cần kernel `ai-gpu` và GPU 12GB cho 7B. Laptop 4GB: đổi `MODEL` sang `Qwen/Qwen2.5-3B-Instruct`
  (hoặc 0.5B) như ghi trong cell cấu hình.
- Chấm: `uv run pytest tracks/ai-engineer/modules/14-fine-tuning-llm/exercises/advanced/01-qlora-7b -q`

In [ ]:
import json
import re

import torch

## 1. Ước lượng trước khi chạy

- `lora_param_count(config, r, targets) -> int`: `config` là dict kiểu `config.json` của Qwen2/Llama (`hidden_size`,
  `intermediate_size`, `num_attention_heads`, `num_key_value_heads`, `num_hidden_layers`, `head_dim` nếu có, không có
  thì `hidden_size // num_attention_heads`). `targets` là tập con của
  `{"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}`. Trả số tham số LoRA hạng `r`.
- `weight_memory_gib(config, linear_bits, other_bits=16) -> float`: bộ nhớ trọng số (GiB) khi mọi lớp tuyến tính
  trong các tầng lưu ở `linear_bits` bit mỗi tham số, còn lại (embedding, `lm_head` nếu không dùng chung với embedding
  theo `tie_word_embeddings`, norm, bias) ở `other_bits`. Bias của q, k, v (Qwen2 có) tính vào phần "còn lại".
- `bnb_4bit_config()`: `BitsAndBytesConfig` cho QLoRA: NF4, double quantization, compute dtype bf16.

In [ ]:
LINEAR = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


def _shapes(config: dict) -> dict[str, tuple[int, int]]:
    # TODO
    raise NotImplementedError


def lora_param_count(config: dict, r: int, targets) -> int:
    # TODO
    raise NotImplementedError


def weight_memory_gib(config: dict, linear_bits: float, other_bits: float = 16) -> float:
    # TODO
    raise NotImplementedError


def bnb_4bit_config():
    # TODO
    raise NotImplementedError

In [ ]:
# explore: ước lượng cho 0.5B, 3B, 7B từ config.json (chỉ tải tệp config, vài KB)
from transformers import AutoConfig

for name in ("Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct"):
    cfg = AutoConfig.from_pretrained(name).to_dict()
    print(f"{name}: LoRA r=16 mọi lớp {lora_param_count(cfg, 16, LINEAR):,} | trọng số bf16 "
          f"{weight_memory_gib(cfg, 16):.2f} GiB | NF4+DQ (4.127 bit) {weight_memory_gib(cfg, 4.127):.2f} GiB")

## 2. Cấu hình QLoRA và hàm dán từ Intermediate 03

- `qlora_sft_kwargs(output_dir, batch_size=4, grad_accum=4, lr=2e-4, epochs=1) -> dict`: tham số cho `SFTConfig(**...)`
  (trả dict vì `SFTConfig` với bf16 báo lỗi trên máy không có GPU): `output_dir`, `per_device_train_batch_size`,
  `gradient_accumulation_steps`, `num_train_epochs`, `learning_rate` theo tham số; `optim="paged_adamw_8bit"`,
  `gradient_checkpointing=True`, `bf16=True`, `max_length=256`, `lr_scheduler_type="cosine"`, `warmup_steps=0.03`,
  `save_strategy="no"`, `report_to=[]`, `seed=0`, `dataloader_num_workers=0`.
- `load_4bit(name)`: nạp model bằng `bnb_4bit_config()` lên GPU 0 (`device_map={"": 0}`), dtype bf16. Hàm này cần GPU
  nên test không gọi nó.
- **Dán** `SYSTEM`, `SENTIMENTS`, `TOPICS`, `to_chat`, `parse_output`, `macro_f1`, `generate_batch` từ bài
  Intermediate 03 (giữ nguyên chữ ký).

In [ ]:
def qlora_sft_kwargs(output_dir: str, batch_size: int = 4, grad_accum: int = 4, lr: float = 2e-4,
                     epochs: int = 1) -> dict:
    # TODO
    raise NotImplementedError


def load_4bit(name: str):
    # TODO
    raise NotImplementedError


SYSTEM = ("Bạn phân loại câu phản hồi của sinh viên về môn học. Trả về đúng một object JSON với hai khoá: "
          '"sentiment" là một trong "negative", "neutral", "positive"; "topic" là một trong "lecturer", '
          '"training_program", "facility", "others". Không viết gì khác.')
SENTIMENTS = ["negative", "neutral", "positive"]
TOPICS = ["lecturer", "training_program", "facility", "others"]


def to_chat(sentence: str, sentiment: str | None = None, topic: str | None = None) -> list[dict]:
    """Dán từ bài Intermediate 03."""
    # TODO
    raise NotImplementedError


def parse_output(text: str, strict: bool = True) -> tuple[str, str] | None:
    """Dán từ bài Intermediate 03."""
    # TODO
    raise NotImplementedError


def macro_f1(y_true: list[str], y_pred: list[str | None], labels: list[str]) -> float:
    """Dán từ bài Intermediate 03."""
    # TODO
    raise NotImplementedError


@torch.no_grad()
def generate_batch(model, tokenizer, chats: list[list[dict]], max_new_tokens: int = 32,
                   batch_size: int = 32) -> list[str]:
    """Dán từ bài Intermediate 03."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: cấu hình chạy. PC 12GB: 7B. Laptop 4GB: "Qwen/Qwen2.5-3B-Instruct" với BATCH 2, GRAD_ACCUM 8
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))


def dataset_dir(name):
    try:
        from data.download_dataset import ensure
    except ImportError:
        path = ROOT / "data" / "raw" / name
        if not path.is_dir():
            raise RuntimeError(f"Chưa có dữ liệu: uv run python data/download_dataset.py {name}")
        return path
    return ensure(name)


VSFC = dataset_dir("uit-vsfc")
MODEL = "Qwen/Qwen2.5-7B-Instruct"
BATCH, GRAD_ACCUM, N_TRAIN = 4, 4, 2000
train = pd.read_parquet(VSFC / "default/train/0000.parquet")
test = pd.read_parquet(VSFC / "default/test/0000.parquet")
pinned = pd.concat([test[test.sentiment == c].sample(50, random_state=0) for c in range(3)]).sample(frac=1.0,
                                                                                                    random_state=0)
rows = train.iloc[np.random.default_rng(0).permutation(len(train))[:N_TRAIN]]


def gold_of(df):
    return [(SENTIMENTS[s], TOPICS[t]) for s, t in zip(df.sentiment, df.topic)]


def evaluate(model, tok, name, batch_size=32):
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    res = {}
    for split, df in (("pinned", pinned), ("test", test)):
        outs = generate_batch(model, tok, [to_chat(s) for s in df.sentence], batch_size=batch_size)
        gold = gold_of(df)
        pred = [parse_output(o) for o in outs]
        res[split] = {"strict_ok": sum(p is not None for p in pred) / len(df),
                      "f1_sentiment": macro_f1([g[0] for g in gold], [p[0] if p else None for p in pred], SENTIMENTS),
                      "f1_topic": macro_f1([g[1] for g in gold], [p[1] if p else None for p in pred], TOPICS)}
    res["seconds"] = round(time.perf_counter() - t0, 1)
    res["peak_gib"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
    print(name, json.dumps(res))
    return res

In [ ]:
# explore: nạp 4 bit, đo trước khi fine-tune (7B tải khoảng 15 GB lần đầu)
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
torch.cuda.reset_peak_memory_stats()
model = load_4bit(MODEL)
print(f"nạp xong: {torch.cuda.memory_allocated() / 2**30:.2f} GiB đã cấp phát, "
      f"get_memory_footprint {model.get_memory_footprint() / 2**30:.2f} GiB")
results = {"model": MODEL, "before": evaluate(model.eval(), tok, "trước")}

In [ ]:
# explore: QLoRA: gradient checkpointing, LoRA r=16 mọi lớp tuyến tính, SFT 1 epoch trên N_TRAIN câu
import tempfile

from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
data = Dataset.from_list([{"prompt": to_chat(s), "completion": to_chat(s, *g)[-1:]}
                          for s, g in zip(rows.sentence, gold_of(rows))])
args = SFTConfig(**qlora_sft_kwargs(tempfile.mkdtemp(prefix="m14-qlora-"), BATCH, GRAD_ACCUM))
trainer = SFTTrainer(model=model, args=args, train_dataset=data, processing_class=tok,
                     peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear",
                                            task_type="CAUSAL_LM"))
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
out = trainer.train()
results["train"] = {"steps": out.global_step, "loss": round(out.training_loss, 4),
                    "seconds": round(time.perf_counter() - t0, 1),
                    "peak_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2)}
print(results["train"])
results["after"] = evaluate(trainer.model.eval(), tok, "sau QLoRA")

## Viết báo cáo

So kết quả với bảng của bài Intermediate 03 (0.5B, prompt và SFT) và chạy thêm 0.5B trên **cùng** `N_TRAIN` câu để so
công bằng. Viết `REPORT.md` theo khung có sẵn, rồi làm phần trắc nghiệm của bài trong app.